# 16 · Foundation Models

Use Prithvi, SAM, SAM2, DINOv2, SatlasPretrain for zero-shot geospatial tasks.

## Contents
1. Foundation model overview
2. NASA Prithvi
3. SAM and GroundedSAM
4. DINOv2 feature extraction
5. Embedding visualisation
6. Transfer learning strategies

## 1 · Foundation Models in the Model Zoo

In [ ]:
from pygeovision.ai.models.zoo import model_zoo
foundation = model_zoo.filter(task='foundation')
print(f'Foundation Models ({len(foundation)}):')
model_zoo.print_table(foundation)

Foundation Models (11):

Name                         Task               Architecture         Params(M)   HF
------------------------------------------------------------------------------------
prithvi_100m                 foundation         Prithvi-100M            100.0    ✓
prithvi_300m                 foundation         Prithvi-300M            300.0    ✓
dinov2_s                     foundation         DINOv2-S                 21.0    ✓
dinov2_b                     foundation         DINOv2-B                 86.0    ✓
dinov2_l                     foundation         DINOv2-L                307.0    ✓
satlas_swin_b                foundation         SatlasPretrain-Swin-B    88.0    ✓
ssl4eo_moco                  foundation         SSL4EO-MoCo              25.0    ✓
dofa_vit_b                   foundation         DOFA-ViT-B               86.0    ✓
geosam_vit_h                 foundation         GeoSAM-ViT-H            636.0    ✓
ringmo_swin_b                foundation         RingMo-Swin

## 2 · NASA Prithvi — Multispectral Foundation

In [ ]:
import pygeovision as pgv
client = pgv.PyGeoVision()

from pygeovision.ai.models.zoo import model_zoo
prithvi_models = [m for m in model_zoo.all() if 'prithvi' in m.name.lower()]
print('Prithvi models available:')
for m in prithvi_models:
    print(f'  {m.name}')
    print(f'    Description: {m.description}')
    print(f'    HF Hub:      {m.hf_model_id}')
    print(f'    Params:      {m.params_m}M | Input bands: {m.input_bands}')
print()
print('Load Prithvi:')
print("  from transformers import AutoModel")
print("  model = AutoModel.from_pretrained('ibm-nasa-geospatial/Prithvi-EO-1.0-100M')")
print()
print('Or via GeoAI:')
print("  prithvi = client.geoai.prithvi.load('prithvi-eo-1.0-100M')")
print("  client.geoai.prithvi.infer('./data/hls.tif', prithvi, task='segmentation')")

Prithvi models available:
  prithvi_100m
    Description: NASA Prithvi HLS 100M MAE
    HF Hub:      ibm-nasa-geospatial/Prithvi-EO-1.0-100M
    Params:      100.0M | Input bands: 6
  prithvi_300m
    Description: NASA Prithvi EO 2.0 300M
    HF Hub:      ibm-nasa-geospatial/Prithvi-EO-2.0-300M
    Params:      300.0M | Input bands: 6


## 3 · SAM and GroundedSAM

In [ ]:
from pygeovision.ai.models.zoo import model_zoo
sam_models = [m for m in model_zoo.all() if 'sam' in m.name.lower()]
print(f'SAM models ({len(sam_models)}):')
for m in sam_models:
    hf = '(HF)' if m.hf_model_id else ''
    print(f'  {m.name:<20} {m.params_m:.0f}M params  {hf}')
print()
print('SAM auto-segmentation:')
print("  client.geoai.sam.generate_masks('./data/aerial.tif',")
print("      output_vector='./results/sam_masks.geojson',")
print("      points_per_side=32, pred_iou_thresh=0.88)")
print()
print('GroundedSAM (text-prompt):')
print("  client.geoai.detect.grounded('./data/aerial.tif', 'swimming pools',")
print("      output_path='./results/pools.geojson')")

SAM models (3):
  sam_vit_h            636M params  (HF)
  sam2_hiera_l         224M params  (HF)
  geosam_vit_h         636M params  (HF)


## 4 · DINOv2 Feature Extraction

In [ ]:
try:
    import torch
    from transformers import AutoImageProcessor, AutoModel
    import numpy as np

    processor = AutoImageProcessor.from_pretrained('facebook/dinov2-large')
    dino = AutoModel.from_pretrained('facebook/dinov2-large')
    dino.eval()
    print('DINOv2-Large loaded (307M params)')

    # Dummy image
    from PIL import Image
    img = Image.fromarray(np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8))
    inputs = processor(images=img, return_tensors='pt')
    with torch.no_grad():
        out = dino(**inputs)
    patches = out.last_hidden_state[:, 1:, :]
    cls     = out.last_hidden_state[:, 0, :]
    print(f'Patches: {tuple(patches.shape)}  -> 256 patches x 1024-dim')
    print(f'CLS:     {tuple(cls.shape)}  -> global image representation')
except ImportError:
    print('Install: pip install transformers torch Pillow')
    print('Usage:')
    print("  from transformers import AutoModel")
    print("  model = AutoModel.from_pretrained('facebook/dinov2-large')")
    print()
    print('Or via GeoAI:')
    print("  result = client.geoai.dinov3.analyze('./data/sentinel2.tif')")
    print("  sim_map = client.geoai.dinov3.create_similarity_map('./data/s2.tif', query_patch_idx=42)")

DINOv2-Large loaded (307M params)
Patches: (1, 256, 1024)  -> 256 patches x 1024-dim
CLS:     (1, 1024)  -> global image representation


## 5 · Transfer Learning Strategies

In [ ]:
strategies = [
    ('Linear Probe',    '~0.1M', '10-30',  'Large datasets, compute constrained',    '0.75-0.80'),
    ('LoRA Fine-tuning','~2-5M', '20-50',  'Medium datasets, best efficiency/acc',   '0.82-0.87'),
    ('Full Fine-tuning','~86M+', '50-100', 'Domain shift, sufficient GPU memory',    '0.85-0.92'),
]
print(f"{'Strategy':<20} {'Params':>8} {'Epochs':>8} {'Best for':<38} {'Expected IoU'}")
print('-' * 95)
for name, params, epochs, best_for, iou in strategies:
    print(f'{name:<20} {params:>8} {epochs:>8} {best_for:<38} {iou}')